# ASG Airlines - Data Engineering Pipeline
## Task 1: Data Ingestion

In [1]:
import pandas as pd
from pathlib import Path
import logging

In [2]:
BASE_DIR = Path.cwd().parent

RAW_FILE = BASE_DIR / "data" / "raw" / "UseCase - Airlines.xlsx"
LOG_DIR = BASE_DIR / "logs"

LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Input file:", RAW_FILE)
print("File exists:", RAW_FILE.exists())

Input file: C:\Users\krkun\OneDrive\Desktop\Neo Stats\data\raw\UseCase - Airlines.xlsx
File exists: True


In [3]:
logging.basicConfig(
    filename=LOG_DIR / "pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("airlines_pipeline")

logger.info("Pipeline started")

In [4]:
try:
    flights = pd.read_excel(
        RAW_FILE,
        sheet_name="flights",
        engine="openpyxl"
    )

    logger.info(
        f"Flights data ingested successfully: {flights.shape}"
    )

    print("Data ingestion successful")
    print("Rows:", flights.shape[0])
    print("Columns:", flights.shape[1])

except FileNotFoundError:
    logger.error("Input Excel file was not found")
    raise

except Exception as e:
    logger.exception("Unexpected error during data ingestion")
    raise

Data ingestion successful
Rows: 1020
Columns: 7


Checking the data

In [5]:
flights.head()

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


In [6]:
flights.tail()

,flight_id,airline,source,destination,departure_time,arrival_time,duration
1015,6F198,IndiGo,HYD,BLR,2026-04-17 12:40:41.703,2026-04-17 17:39:41.703,04:59:00
1016,AI093,Air India,BOM,BLR,2026-04-17 12:35:41.702,2026-04-17 17:28:41.702,04:53:00
1017,UK062,Vistara,HYD,DEL,2026-04-17 12:33:41.701,2026-04-17 15:35:41.701,03:02:00
1018,6F057,IndiGo,MAA,DEL,2026-04-17 12:32:41.701,2026-04-17 15:50:41.701,03:18:00
1019,UK064,Vistara,BOM,DEL,2026-04-17 12:25:41.701,2026-04-17 14:55:41.701,02:30:00


In [7]:
flights.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   flight_id       1020 non-null   str           
 1   airline         979 non-null    str           
 2   source          1020 non-null   str           
 3   destination     1020 non-null   str           
 4   departure_time  1020 non-null   datetime64[us]
 5   arrival_time    1020 non-null   datetime64[us]
 6   duration        1020 non-null   object        
dtypes: datetime64[us](2), object(1), str(4)
memory usage: 74.1+ KB


Validate the columns

In [8]:
expected_columns = [
    "flight_id",
    "airline",
    "source",
    "destination",
    "departure_time",
    "arrival_time",
    "duration"
]

actual_columns = flights.columns.tolist()

missing_columns = [
    col for col in expected_columns
    if col not in actual_columns
]

unexpected_columns = [
    col for col in actual_columns
    if col not in expected_columns
]

print("Missing columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

Missing columns: []
Unexpected columns: []


Check row count

In [9]:
row_count = len(flights)

print("Total flight records:", row_count)

if row_count == 0:
    logger.error("Flight dataset contains zero records")
    raise ValueError("Empty flight dataset")

logger.info(f"Validated row count: {row_count}")

Total flight records: 1020


Check missing values

In [10]:
missing_values = flights.isnull().sum()

print(missing_values)

flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64


In [11]:
quality_report = pd.DataFrame({
    "column": flights.columns,
    "data_type": flights.dtypes.astype(str).values,
    "missing_count": flights.isnull().sum().values,
    "missing_percentage": (
        flights.isnull().sum().values / len(flights) * 100
    )
})

quality_report

,column,data_type,missing_count,missing_percentage
0,flight_id,str,0,0.000000
1,airline,str,41,4.019608
2,source,str,0,0.000000
3,destination,str,0,0.000000
4,departure_time,datetime64[us],0,0.000000
5,arrival_time,datetime64[us],0,0.000000
6,duration,object,0,0.000000


Check duplicate records

In [12]:
duplicate_rows = flights.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

logger.info(
    f"Duplicate rows detected: {duplicate_rows}"
)

Duplicate rows: 15


Check duplicate flight IDs

In [13]:
duplicate_ids = flights[
    flights["flight_id"].duplicated(keep=False)
]

print("Records with duplicate flight IDs:",
      len(duplicate_ids))

Records with duplicate flight IDs: 32


In [14]:
flights["flight_id"].head(20)

0     SJ010
1     AI155
2     UK094
3     AI245
4     AI192
5     SJ158
6     6F196
7     AI080
8     6F025
9     6F251
10    AI137
11    6F026
12    UK167
13    UK027
14    AI069
15    AI058
16    AI140
17    SJ239
18    6F099
19    AI048
Name: flight_id, dtype: str

In [15]:
flights["flight_id"].unique()

<ArrowStringArray>
['SJ010', 'AI155', 'UK094', 'AI245', 'AI192', 'SJ158', '6F196', 'AI080',
 '6F025', '6F251',
 ...
 'AI052', 'SJ177', '6F215', 'AI223', 'SJ069', '6F198', 'AI093', 'UK062',
 '6F057', 'UK064']
Length: 1004, dtype: str

AI155 \
││└── 3 digits\
└┴── airline code

In [16]:
flight_id_pattern = r"^(AI|UK|SJ|6F)\d{3}$"

valid_flight_ids = flights["flight_id"].astype("string").str.match(
    flight_id_pattern,
    na=False
)

print("Valid flight IDs:", valid_flight_ids.sum())
print("Invalid flight IDs:", (~valid_flight_ids).sum())

Valid flight IDs: 1020
Invalid flight IDs: 0


Check time consistency

In [17]:
arrival_before_departure = (
    flights["arrival_time"] < flights["departure_time"]
)

print(
    "Arrival before departure:",
    arrival_before_departure.sum()
)

Arrival before departure: 1


In [18]:
flights["duration"].head(20)

0     02:54:00
1     01:48:00
2     01:45:00
3     02:36:00
4     04:59:00
5     02:19:00
6     01:43:00
7     01:32:00
8     00:55:00
9     00:41:00
10    02:28:00
11    04:27:00
12    03:01:00
13    02:21:00
14    03:46:00
15    01:38:00
16    03:43:00
17    01:46:00
18    00:35:00
19    03:52:00
Name: duration, dtype: object

Save the raw ingested data

In [19]:
RAW_OUTPUT = BASE_DIR / "data" / "raw" / "flights_ingested.csv"

flights.to_csv(
    RAW_OUTPUT,
    index=False
)

logger.info(
    f"Raw ingested data saved to {RAW_OUTPUT}"
)

print("Saved:", RAW_OUTPUT)

Saved: C:\Users\krkun\OneDrive\Desktop\Neo Stats\data\raw\flights_ingested.csv


Add a final ingestion summary

In [20]:
print("=" * 50)
print("DATA INGESTION SUMMARY")
print("=" * 50)

print(f"Source file       : {RAW_FILE.name}")
print(f"Source sheet      : flights")
print(f"Records ingested  : {len(flights)}")
print(f"Columns           : {len(flights.columns)}")
print(f"Missing values    : {flights.isnull().sum().sum()}")
print(f"Duplicate rows    : {flights.duplicated().sum()}")
print(f"Output file       : {RAW_OUTPUT}")

print("=" * 50)

logger.info("Data ingestion completed successfully")

DATA INGESTION SUMMARY
Source file       : UseCase - Airlines.xlsx
Source sheet      : flights
Records ingested  : 1020
Columns           : 7
Missing values    : 41
Duplicate rows    : 15
Output file       : C:\Users\krkun\OneDrive\Desktop\Neo Stats\data\raw\flights_ingested.csv


# Task 2: Data Cleaning and Transformation

In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

In [22]:
BASE_DIR = Path.cwd().parent

INPUT_FILE = BASE_DIR / "data" / "raw" / "flights_ingested.csv"

flights = pd.read_csv(
    INPUT_FILE,
    parse_dates=["departure_time", "arrival_time"]
)

print(flights.shape)

(1020, 7)


In [23]:
df = flights.copy()

In [24]:
df

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00
...,...,...,...,...,...,...,...
1015,6F198,IndiGo,HYD,BLR,2026-04-17 12:40:41.703,2026-04-17 17:39:41.703,04:59:00
1016,AI093,Air India,BOM,BLR,2026-04-17 12:35:41.702,2026-04-17 17:28:41.702,04:53:00
1017,UK062,Vistara,HYD,DEL,2026-04-17 12:33:41.701,2026-04-17 15:35:41.701,03:02:00
1018,6F057,IndiGo,MAA,DEL,2026-04-17 12:32:41.701,2026-04-17 15:50:41.701,03:18:00


In [25]:
df.duplicated().sum()

np.int64(15)

In [26]:
df = df.drop_duplicates().copy()

In [27]:
print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1005


Clean text columns

In [28]:
text_columns = [
    "flight_id",
    "airline",
    "source",
    "destination"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

Standardize airline names

In [29]:
# NOTE: previously used .str.title(), which corrupts already-correct
# source spellings ("IndiGo" -> "Indigo", "SpiceJet" -> "Spicejet",
# "UNKNOWN" -> "Unknown"). That mismatch is why the same airline later
# showed up under two different spellings. The source only ever contains
# these exact spellings (verified against the raw workbook), so we
# normalize case-insensitively onto the real spellings instead of
# blindly title-casing.
canonical_airlines = {
    "indigo": "IndiGo",
    "spicejet": "SpiceJet",
    "air india": "Air India",
    "vistara": "Vistara",
    "unknown": "Unknown"
}

df["airline"] = df["airline"].str.strip().apply(
    lambda v: canonical_airlines.get(v.lower(), v) if pd.notna(v) else v
)

In [30]:
df["airline"].value_counts(dropna=False)

airline
IndiGo       249
SpiceJet     236
Air India    233
Vistara      218
NaN           39
Unknown       30
Name: count, dtype: int64

Standardize airport codes

In [31]:
df["source"] = df["source"].str.strip().str.upper()
df["destination"] = df["destination"].str.strip().str.upper()

In [32]:
print(df["source"].unique())
print(df["destination"].unique())

<ArrowStringArray>
['CCU', 'BOM', 'MAA', 'DEL', 'BLR', 'HYD']
Length: 6, dtype: string
<ArrowStringArray>
['MAA', 'CCU', 'BOM', 'HYD', 'BLR', 'DEL']
Length: 6, dtype: string


Handle missing airline values

In [33]:
missing_airline = df[df["airline"].isna()]

missing_airline[
    ["flight_id", "airline", "source", "destination"]
].head(50)

,flight_id,airline,source,destination
57,UK209,NaN,MAA,BLR
63,6F270,NaN,HYD,MAA
89,UK224,NaN,CCU,MAA
118,6F261,NaN,CCU,DEL
120,6F254,NaN,MAA,BLR
124,6F259,NaN,BOM,DEL
133,UK141,NaN,HYD,MAA
150,6F255,NaN,DEL,BOM
218,AI047,NaN,CCU,MAA
222,AI128,NaN,BOM,CCU


In [34]:
df[
    df["flight_id"].notna()
].assign(
    prefix=df["flight_id"].str[:2]
).groupby("prefix")["airline"].value_counts(dropna=False)

prefix  airline  
6F      IndiGo       249
        Unknown       12
        NaN           12
AI      Air India    233
        NaN           14
        Unknown        8
SJ      SpiceJet     236
        Unknown        6
        NaN            5
UK      Vistara      218
        NaN            8
        Unknown        4
Name: count, dtype: int64

Build an airline prefix mapping

In [35]:
#   6F -> IndiGo, AI -> Air India, SJ -> SpiceJet, UK -> Vistara
airline_prefix_map = {
    "AI": "Air India",
    "UK": "Vistara",
    "SJ": "SpiceJet",
    "6F": "IndiGo"
}

In [36]:
def infer_airline(row):
    airline = row["airline"]
    if pd.notna(airline) and airline != "Unknown":
        return airline

    flight_id = row["flight_id"]

    if pd.isna(flight_id):
        return "Unknown"

    prefix = str(flight_id)[:2]

    return airline_prefix_map.get(prefix, "Unknown")

In [37]:
df["airline"] = df.apply(
    infer_airline,
    axis=1
)

In [38]:
df["airline"].isna().sum()

np.int64(0)

In [39]:
df["airline"] = df["airline"].fillna("Unknown")

Clean the Flight ID

In [40]:
def get_pattern(value):
    if pd.isna(value):
        return "MISSING"

    value = str(value)

    pattern = ""

    for char in value:
        if char.isalpha():
            pattern += "A"
        elif char.isdigit():
            pattern += "9"
        else:
            pattern += "X"

    return pattern

In [41]:
df["flight_id_pattern"] = df["flight_id"].apply(
    get_pattern
)

In [42]:
df["flight_id_pattern"].value_counts()

flight_id_pattern
AA999    732
9A999    273
Name: count, dtype: int64

Validate Flight IDs

In [43]:
valid_pattern = r"^(AA999|9A999)$"

df["flight_id_valid"] = (
    df["flight_id_pattern"]
    .str.match(valid_pattern, na=False)
)

In [44]:
df["flight_id_valid"].value_counts()

flight_id_valid
True    1005
Name: count, dtype: int64

Handle duplicate Flight IDs

In [45]:
df["flight_id"].duplicated().sum()

np.int64(1)

In [46]:
duplicate_ids = df[
    df["flight_id"].duplicated(keep=False)
].sort_values("flight_id")

duplicate_ids

,flight_id,airline,source,destination,departure_time,arrival_time,duration,flight_id_pattern,flight_id_valid
253,6F250,IndiGo,DEL,BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701,04:04:00,9A999,True
270,6F250,IndiGo,CCU,BLR,2026-04-20 02:23:41.702,2026-04-20 02:56:41.702,00:33:00,9A999,True


The same flight_id 6F250 appears twice, but the flights have different sources:

6F250 → DEL → BLR
6F250 → CCU → BLR

So these are not exact duplicate rows. so we will not delete them 

In [47]:
df[df["flight_id"] == "6F250"]

,flight_id,airline,source,destination,departure_time,arrival_time,duration,flight_id_pattern,flight_id_valid
253,6F250,IndiGo,DEL,BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701,04:04:00,9A999,True
270,6F250,IndiGo,CCU,BLR,2026-04-20 02:23:41.702,2026-04-20 02:56:41.702,00:33:00,9A999,True


create a flag: as flight id is same but other things are different .

In [48]:
df["duplicate_flight_id_flag"] = (
    df["flight_id"].duplicated(keep=False)
)

In [49]:
df[
    df["duplicate_flight_id_flag"]
]

,flight_id,airline,source,destination,departure_time,arrival_time,duration,flight_id_pattern,flight_id_valid,duplicate_flight_id_flag
253,6F250,IndiGo,DEL,BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701,04:04:00,9A999,True,True
270,6F250,IndiGo,CCU,BLR,2026-04-20 02:23:41.702,2026-04-20 02:56:41.702,00:33:00,9A999,True,True


checking whether the entire flight information is duplicated

In [50]:
duplicate_flight_records = df.duplicated(
    subset=[
        "flight_id",
        "source",
        "destination",
        "departure_time",
        "arrival_time"
    ],
    keep=False
)

print(
    "Duplicate flight records:",
    duplicate_flight_records.sum()
)

Duplicate flight records: 0


checking times are datetime

In [51]:
df["departure_time"] = pd.to_datetime(
    df["departure_time"],
    errors="coerce"
)

df["arrival_time"] = pd.to_datetime(
    df["arrival_time"],
    errors="coerce"
)

In [52]:
print(df["departure_time"].dtype)
print(df["arrival_time"].dtype)

datetime64[us]
datetime64[us]


Create departure date

In [53]:
df["departure_date"] = (
    df["departure_time"].dt.date
)

In [54]:
df["departure_year"] = (
    df["departure_time"].dt.year
)

df["departure_month"] = (
    df["departure_time"].dt.month
)

df["departure_day"] = (
    df["departure_time"].dt.day
)

Calculating whether the flight is overnight.

In [55]:
df["overnight_flag"] = (
    df["arrival_time"].dt.date >
    df["departure_time"].dt.date
)

In [56]:
df["overnight_flag"].value_counts()

overnight_flag
False    883
True     122
Name: count, dtype: int64

Calculate flight duration

In [57]:
df["flight_duration_minutes"] = (
    df["arrival_time"] -
    df["departure_time"]
).dt.total_seconds() / 60

# (KPIs, exports) does not need to change.
df["calculated_duration_minutes"] = df["flight_duration_minutes"]


In [58]:
df[
    [
        "flight_id",
        "departure_time",
        "arrival_time",
        "flight_duration_minutes",
        "overnight_flag"
    ]
].head(20)

,flight_id,departure_time,arrival_time,flight_duration_minutes,overnight_flag
0,SJ010,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,174.0,True
1,AI155,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,108.0,True
2,UK094,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,105.0,True
3,AI245,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,156.0,True
4,AI192,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,299.0,True
5,SJ158,2026-04-20 23:05:41.703,2026-04-21 01:24:41.703,139.0,True
6,6F196,2026-04-20 23:04:41.703,2026-04-21 00:47:41.703,103.0,True
7,AI080,2026-04-20 23:03:41.702,2026-04-21 00:35:41.702,92.0,True
8,6F025,2026-04-20 23:02:41.701,2026-04-20 23:57:41.701,55.0,False
9,6F251,2026-04-20 22:56:41.703,2026-04-20 23:37:41.703,41.0,False


Validate the duration

In [59]:
negative_duration = df[
    df["flight_duration_minutes"] < 0
]

print(
    "Negative durations:",
    len(negative_duration)
)

Negative durations: 1


Compare source duration vs calculated duration

In [60]:
import datetime as dt
EXCEL_DURATION_EPOCH = dt.datetime(1899, 12, 30)


def parse_duration_value(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, pd.Timedelta):
        return value
    if isinstance(value, dt.time):
        return pd.Timedelta(
            hours=value.hour, minutes=value.minute, seconds=value.second
        )
    if isinstance(value, dt.datetime):
        return value - EXCEL_DURATION_EPOCH
    if isinstance(value, (int, float)):
        return pd.Timedelta(days=value)
    try:
        return pd.to_timedelta(value)
    except (ValueError, TypeError):
        return pd.NaT


df["source_duration"] = df["duration"].apply(parse_duration_value)


In [61]:
df["source_duration_minutes"] = (
    df["source_duration"]
    .dt.total_seconds() / 60
)

In [62]:
df["duration_difference"] = (
    df["flight_duration_minutes"]
    - df["source_duration_minutes"]
)

In [63]:
df[
    df["duration_difference"].abs() > 1
][
    [
        "flight_id",
        "departure_time",
        "arrival_time",
        "duration",
        "flight_duration_minutes",
        "source_duration_minutes"
    ]
].head(20)

,flight_id,departure_time,arrival_time,duration,flight_duration_minutes,source_duration_minutes


Create Route

In [64]:
df["route"] = (
    df["source"] +
    " → " +
    df["destination"]
)

Flag missing values

In [65]:
df["missing_time_flag"] = (
    df["departure_time"].isna() |
    df["arrival_time"].isna()
)

In [66]:
df["missing_route_flag"] = (
    df["source"].isna() |
    df["destination"].isna()
)

Flag invalid duration

In [67]:
df["invalid_duration_flag"] = (
    df["flight_duration_minutes"].isna() |
    (df["flight_duration_minutes"] <= 0)
)

In [68]:
df["invalid_flight_id_flag"] = ~df["flight_id_valid"]
df["missing_airline_flag"] = df["airline"] == "Unknown"
df["missing_source_flag"] = df["source"].isna()
df["missing_destination_flag"] = df["destination"].isna()
df["missing_departure_time_flag"] = df["departure_time"].isna()
df["missing_arrival_time_flag"] = df["arrival_time"].isna()

# Impossible timestamp ordering: arrival at/before departure. SJ192 is the only occurrence in this dataset, but the check itself is general.
df["timestamp_order_flag"] = (
    df["departure_time"].notna() &
    df["arrival_time"].notna() &
    (df["arrival_time"] <= df["departure_time"])
)

# normal overnight flight.
df["sj192_anomaly_flag"] = df["flight_id"] == "SJ192"

df["duration_anomaly_flag"] = (
    df["source_duration_minutes"].isna() |
    df["calculated_duration_minutes"].isna() |
    (df["calculated_duration_minutes"] <= 0) |
    (df["source_duration_minutes"] <= 0) |
    df["timestamp_order_flag"] |
    df["sj192_anomaly_flag"]
)
df["duration_discrepancy_flag"] = df["duration_difference"].abs() > 1

df["referential_integrity_flag"] = df["duplicate_flight_id_flag"]

df["duration_kpi_eligible_flag"] = ~(
    df["missing_departure_time_flag"] |
    df["missing_arrival_time_flag"] |
    df["timestamp_order_flag"] |
    df["sj192_anomaly_flag"] |
    df["duration_anomaly_flag"]
)

# flight_id is not guaranteed unique (6F250 identifies two distinct flight
# events, see below).
df["flight_event_key"] = (
    df["airline"].astype("string") + "|" +
    df["flight_id"].astype("string") + "|" +
    df["source"].astype("string") + "|" +
    df["destination"].astype("string") + "|" +
    df["departure_time"].astype("string")
)

granular_flags = [
    "invalid_flight_id_flag", "missing_airline_flag", "missing_source_flag",
    "missing_destination_flag", "missing_departure_time_flag",
    "missing_arrival_time_flag", "duplicate_flight_id_flag", "overnight_flag",
    "timestamp_order_flag", "sj192_anomaly_flag", "duration_anomaly_flag",
    "duration_discrepancy_flag", "referential_integrity_flag",
    "duration_kpi_eligible_flag"
]
df[granular_flags].sum()


invalid_flight_id_flag            0.0
missing_airline_flag              0.0
missing_source_flag               0.0
missing_destination_flag          0.0
missing_departure_time_flag       0.0
missing_arrival_time_flag         0.0
duplicate_flight_id_flag          2.0
overnight_flag                  122.0
timestamp_order_flag              1.0
sj192_anomaly_flag                1.0
duration_anomaly_flag             1.0
duration_discrepancy_flag         0.0
referential_integrity_flag        2.0
duration_kpi_eligible_flag     1004.0
dtype: double[pyarrow]

Creating an overall anomaly flag

In [69]:
df["anomaly_flag"] = (
    df["invalid_flight_id_flag"] |
    df["duplicate_flight_id_flag"] |
    df["missing_airline_flag"] |
    df["missing_source_flag"] |
    df["missing_destination_flag"] |
    df["missing_departure_time_flag"] |
    df["missing_arrival_time_flag"] |
    df["timestamp_order_flag"] |
    df["sj192_anomaly_flag"] |
    df["duration_anomaly_flag"] |
    df["duration_discrepancy_flag"] |
    df["referential_integrity_flag"]
)


In [70]:
df["anomaly_flag"].value_counts()

anomaly_flag
False    1002
True        3
Name: count, dtype: int64[pyarrow]

In [71]:
def get_anomaly_reason(row):

    reasons = []

    if row["invalid_flight_id_flag"]:
        reasons.append("Invalid Flight ID")

    if row["duplicate_flight_id_flag"]:
        reasons.append("Duplicate Flight ID")

    if row["missing_airline_flag"]:
        reasons.append("Missing Airline")

    if row["missing_source_flag"]:
        reasons.append("Missing Source")

    if row["missing_destination_flag"]:
        reasons.append("Missing Destination")

    if row["missing_departure_time_flag"]:
        reasons.append("Missing Departure Time")

    if row["missing_arrival_time_flag"]:
        reasons.append("Missing Arrival Time")

    if row["sj192_anomaly_flag"]:
        reasons.append("SJ192 Impossible Timestamp Anomaly")
    elif row["timestamp_order_flag"]:
        reasons.append("Impossible Timestamp Ordering")

    if row["duration_anomaly_flag"]:
        reasons.append("Invalid Duration")

    if row["duration_discrepancy_flag"]:
        reasons.append("Duration Discrepancy")

    if row["referential_integrity_flag"]:
        reasons.append("Referential Integrity Failure")

    if reasons:
        return ", ".join(reasons)

    return "Valid"


In [72]:
df["anomaly_reason"] = df.apply(
    get_anomaly_reason,
    axis=1
)

In [73]:
df["anomaly_reason"].value_counts()

anomaly_reason
Valid                                                   1002
Duplicate Flight ID, Referential Integrity Failure         2
SJ192 Impossible Timestamp Anomaly, Invalid Duration       1
Name: count, dtype: int64

Separate clean and rejected/flagged data

In [74]:
fact_exclusion_flag = (
    df["invalid_flight_id_flag"] |
    df["missing_airline_flag"] |
    df["missing_source_flag"] |
    df["missing_destination_flag"] |
    df["missing_departure_time_flag"] |
    df["missing_arrival_time_flag"] |
    df["timestamp_order_flag"] |
    df["sj192_anomaly_flag"] |
    df["duration_anomaly_flag"]
)

clean_flights = df[~fact_exclusion_flag].copy()

anomaly_flights = df[
    df["anomaly_flag"]
].copy()


In [75]:
print("Total:", len(df))
print("Clean (fact-eligible):", len(clean_flights))
print("Anomalies (flagged for review):", len(anomaly_flights))
print(
    "Note: these two counts can overlap. e.g. the two 6F250 records are "
    "valid flight facts (kept in clean_flights) that are also flagged for "
    "traceability (duplicate_flight_id_flag) in anomaly_flights."
)


Total: 1005
Clean (fact-eligible): 1004
Anomalies (flagged for review): 3
Note: these two counts can overlap. e.g. the two 6F250 records are valid flight facts (kept in clean_flights) that are also flagged for traceability (duplicate_flight_id_flag) in anomaly_flights.


In [76]:
final_columns = [
    "flight_event_key",
    "flight_id",
    "airline",
    "source",
    "destination",
    "route",
    "departure_time",
    "arrival_time",
    "departure_date",
    "flight_duration_minutes",
    "calculated_duration_minutes",
    "source_duration_minutes",
    "duration_difference",
    "overnight_flag",
    "duplicate_flight_id_flag",
    "invalid_flight_id_flag",
    "missing_airline_flag",
    "missing_source_flag",
    "missing_destination_flag",
    "missing_departure_time_flag",
    "missing_arrival_time_flag",
    "timestamp_order_flag",
    "sj192_anomaly_flag",
    "duration_anomaly_flag",
    "duration_discrepancy_flag",
    "referential_integrity_flag",
    "duration_kpi_eligible_flag",
    "anomaly_flag",
    "anomaly_reason"
]

clean_flights = clean_flights[
    final_columns
].copy()


Save your clean dataset

In [77]:
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [78]:
clean_flights.to_csv(
    PROCESSED_DIR / "clean_flights.csv",
    index=False
)

In [79]:
anomaly_flights.to_csv(
    PROCESSED_DIR / "flight_anomalies.csv",
    index=False
)

Final validation

In [80]:
print("Final clean dataset")
print("=" * 40)

print("Rows:", len(clean_flights))
print("Columns:", len(clean_flights.columns))
print(
    "Missing values:",
    clean_flights.isna().sum().sum()
)
print(
    "Duplicate rows:",
    clean_flights.duplicated().sum()
)
print(
    "Negative durations:",
    (
        clean_flights["flight_duration_minutes"] < 0
    ).sum()
)

Final clean dataset
Rows: 1004
Columns: 29
Missing values: 0
Duplicate rows: 0
Negative durations: 0


# Task 2 (continued): Other Data Sheets — Bookings, Payments, Passengers

Task 2 applies to the whole dataset, not only `flights`. Before applying any
cleaning rule we ingest and profile `bookings`, `payments`, and `passengers`
directly from the source workbook.

In [81]:
try:
    bookings = pd.read_excel(RAW_FILE, sheet_name="bookings", engine="openpyxl")
    payments = pd.read_excel(RAW_FILE, sheet_name="payments", engine="openpyxl")
    passengers = pd.read_excel(RAW_FILE, sheet_name="passengers", engine="openpyxl")

    logger.info(f"Bookings sheet loaded: {bookings.shape}")
    logger.info(f"Payments sheet loaded: {payments.shape}")
    logger.info(f"Passengers sheet loaded: {passengers.shape}")

except Exception as e:
    logger.exception("Unexpected error while loading bookings/payments/passengers")
    raise

print("bookings   :", bookings.shape)
print("payments   :", payments.shape)
print("passengers :", passengers.shape)

bookings   : (1000, 9)
payments   : (1000, 4)
passengers : (1039, 9)


## Data Quality Profile — Bookings, Payments, Passengers

In [82]:
print("=" * 50)
print("BOOKINGS")
print("=" * 50)
print(bookings.isnull().sum())
print("Exact duplicate rows:", bookings.duplicated().sum())
print("Duplicate booking_id:", bookings["booking_id"].duplicated().sum())
print(bookings["status"].value_counts(dropna=False))

print()
print("=" * 50)
print("PAYMENTS")
print("=" * 50)
print(payments.isnull().sum())
print("Exact duplicate rows:", payments.duplicated().sum())
print("Duplicate payment_id:", payments["payment_id"].duplicated().sum())
print("amount value types:")
print(payments["amount"].apply(lambda v: type(v).__name__).value_counts())
print("Literal INVALID amounts:", (payments["amount"] == "INVALID").sum())
print(payments["payment_method"].value_counts(dropna=False))

print()
print("=" * 50)
print("PASSENGERS")
print("=" * 50)
print(passengers.isnull().sum())
print("Exact duplicate rows:", passengers.duplicated().sum())
print("Duplicate passenger_id:", passengers["passenger_id"].duplicated().sum())
print(passengers["gender"].value_counts(dropna=False))

aadhaar_len = passengers["aadhaar_id"].astype(str).str.len()
print("Aadhaar digit-length distribution:")
print(aadhaar_len.value_counts().sort_index())

BOOKINGS
booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64
Exact duplicate rows: 0
Duplicate booking_id: 0
status
CONFIRMED    320
CANCELLED    314
PENDING      291
NaN           45
INVALID       30
Name: count, dtype: int64

PAYMENTS
payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64
Exact duplicate rows: 0
Duplicate payment_id: 0
amount value types:
amount
float    961
str       30
int        9
Name: count, dtype: int64
Literal INVALID amounts: 30
payment_method
UPI           358
CARD          329
NETBANKING    313
Name: count, dtype: int64

PASSENGERS
passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0


In [83]:
# The duplicate passenger_id count above is unexpected 
dup_pax_ids = passengers.loc[
    passengers["passenger_id"].duplicated(keep=False), "passenger_id"
].unique()

identity_cols = [
    "first_name", "last_name", "age", "gender",
    "email", "phone", "aadhaar_id", "date_of_birth"
]

conflicting_groups = 0
for pid, g in passengers.groupby("passenger_id"):
    if len(g) > 1 and g[identity_cols].drop_duplicates().shape[0] > 1:
        conflicting_groups += 1

print("Distinct passenger_id values that repeat:", len(dup_pax_ids))
print("Of those, groups where the repeated rows are genuinely different people:",
      conflicting_groups)
print("Total passenger rows involved:", passengers["passenger_id"].isin(dup_pax_ids).sum())
print("Bookings that reference one of these ambiguous IDs:",
      bookings["passenger_id"].isin(dup_pax_ids).sum())

passengers[passengers["passenger_id"].isin(dup_pax_ids[:1])][
    ["passenger_id", "first_name", "last_name", "aadhaar_id", "date_of_birth"]
]

Distinct passenger_id values that repeat: 36
Of those, groups where the repeated rows are genuinely different people: 36
Total passenger rows involved: 75
Bookings that reference one of these ambiguous IDs: 34


,passenger_id,first_name,last_name,aadhaar_id,date_of_birth
34,P1034,Ishaan,Iyer,122362316658,1964-08-14
35,P1034,Ishaan,Iyer,669096705466,1964-09-24


### What we found and the treatment decisions

**RECONSTRUCTED NOTE:** the original version of this markdown cell was lost
during a later `nbconvert --execute` re-run and could not be recovered
verbatim. This is a paraphrase of the same decisions, written to match what
the code in the cells below actually does -- please review it for accuracy
rather than treating it as the original wording.

**Bookings -- `status`**: some rows have a missing status or the literal
string `"INVALID"`. Both are recoded to `"Unknown"` rather than dropped
(the booking itself is still real), and flagged via `missing_status_flag`
and `invalid_status_flag` so they remain auditable.

**Payments -- `amount`**: some rows have a missing amount or the literal
string `"INVALID"` instead of a number. `pd.to_numeric(..., errors="coerce")`
converts these to `NaN` rather than fabricating a value; `missing_amount_flag`
and `invalid_amount_flag` record which case applied. Revenue KPIs computed
later exclude these `NaN` rows instead of treating them as zero.

**Passengers -- duplicate `passenger_id`**: `passenger_id` repeats for more
rows than expected. Before deciding how to treat them, we checked whether
the repeated rows are the same person (an ingestion duplicate) or different
people who happen to share an ID (a genuine identity collision) by comparing
name/age/gender/email/phone/aadhaar/date_of_birth across each group. The
repeated groups turned out to be genuinely different people, so they are
**not deduplicated** -- that would silently merge two real passengers.
Instead they are quarantined into `passenger_anomalies` and excluded from
the clean `passengers` dimension, which keeps `passenger_id` unique for
joins. Bookings that reference one of these ambiguous IDs are flagged via
`ambiguous_passenger_flag` rather than silently matched to the wrong person.


In [84]:
df_bookings = bookings.copy()

for col in ["booking_id", "passenger_id", "flight_id", "status",
            "passport_number", "seat_number",
            "emergency_contact_name", "emergency_contact_phone"]:
    df_bookings[col] = df_bookings[col].astype("string").str.strip()

df_bookings["missing_status_flag"] = df_bookings["status"].isna()
df_bookings["invalid_status_flag"] = df_bookings["status"] == "INVALID"

df_bookings["status"] = df_bookings["status"].where(
    ~(df_bookings["missing_status_flag"] | df_bookings["invalid_status_flag"]),
    "Unknown"
)

print(df_bookings["status"].value_counts(dropna=False))
print()
print("missing_status_flag:", df_bookings["missing_status_flag"].sum())
print("invalid_status_flag:", df_bookings["invalid_status_flag"].sum())
logger.info(
    f"Bookings status cleaned: "
    f"{df_bookings['missing_status_flag'].sum()} missing, "
    f"{df_bookings['invalid_status_flag'].sum()} invalid recoded to Unknown"
)

status
CONFIRMED    320
CANCELLED    314
PENDING      291
Unknown       75
Name: count, dtype: int64[pyarrow]

missing_status_flag: 45
invalid_status_flag: 30


In [85]:
df_payments = payments.copy()

for col in ["payment_id", "booking_id", "payment_method"]:
    df_payments[col] = df_payments[col].astype("string").str.strip()

df_payments["missing_amount_flag"] = df_payments["amount"].isna()
df_payments["invalid_amount_flag"] = df_payments["amount"].apply(
    lambda v: isinstance(v, str)
)

df_payments["amount"] = pd.to_numeric(df_payments["amount"], errors="coerce")

print("missing_amount_flag:", df_payments["missing_amount_flag"].sum())
print("invalid_amount_flag:", df_payments["invalid_amount_flag"].sum())
print("Valid numeric amounts:", df_payments["amount"].notna().sum())
print(df_payments["amount"].describe())

logger.info(
    f"Payments amount cleaned: "
    f"{df_payments['missing_amount_flag'].sum()} missing, "
    f"{df_payments['invalid_amount_flag'].sum()} invalid, "
    f"{df_payments['amount'].notna().sum()} valid numeric"
)

missing_amount_flag: 48
invalid_amount_flag: 30
Valid numeric amounts: 922
count      922.000000
mean      8009.916464
std       4022.055392
min       1002.590000
25%       4623.495000
50%       8027.125000
75%      11288.155000
max      14992.950000
Name: amount, dtype: float64


In [86]:
df_passengers = passengers.copy()

for col in ["passenger_id", "first_name", "last_name", "gender", "email", "phone"]:
    df_passengers[col] = df_passengers[col].astype("string").str.strip()

df_passengers["missing_last_name_flag"] = df_passengers["last_name"].isna()

# Restore the digit count Python's int64 silently strips
df_passengers["aadhaar_id_padded"] = (
    df_passengers["aadhaar_id"].astype(str).str.zfill(12)
)

# Flag passenger_id values that map to more than one genuinely different person.
dup_pax_ids = df_passengers.loc[
    df_passengers["passenger_id"].duplicated(keep=False), "passenger_id"
].unique()

df_passengers["ambiguous_passenger_id_flag"] = df_passengers["passenger_id"].isin(
    dup_pax_ids
)

clean_passengers = df_passengers[~df_passengers["ambiguous_passenger_id_flag"]].copy()
passenger_anomalies = df_passengers[df_passengers["ambiguous_passenger_id_flag"]].copy()

print("missing_last_name_flag:", df_passengers["missing_last_name_flag"].sum())
print("Total passengers:", len(df_passengers))
print("Clean passengers (unique, unambiguous ID):", len(clean_passengers))
print("Quarantined passenger anomalies (ambiguous ID):", len(passenger_anomalies))
print("clean_passengers passenger_id now unique:",
      clean_passengers["passenger_id"].is_unique)

logger.info(
    f"Passengers cleaned: {len(clean_passengers)} clean, "
    f"{len(passenger_anomalies)} quarantined (ambiguous passenger_id)"
)

missing_last_name_flag: 10
Total passengers: 1039
Clean passengers (unique, unambiguous ID): 964
Quarantined passenger anomalies (ambiguous ID): 75


clean_passengers passenger_id now unique: True


## Referential Integrity

In [87]:
orphan_flight = ~df_bookings["flight_id"].isin(df["flight_id"])
orphan_passenger = ~df_bookings["passenger_id"].isin(passengers["passenger_id"])
orphan_booking = ~df_payments["booking_id"].isin(df_bookings["booking_id"])

print("bookings.flight_id orphans:", orphan_flight.sum())
print("bookings.passenger_id orphans:", orphan_passenger.sum())
print("payments.booking_id orphans:", orphan_booking.sum())
print(
    "-> No orphans on any of the three relationships; "
    "nothing to quarantine for missing parents."
)
# ambiguous rather than missing:
ambiguous_flight_ids = df.loc[df["duplicate_flight_id_flag"], "flight_id"].unique()
ambiguous_passenger_ids = passenger_anomalies["passenger_id"].unique()

df_bookings["ambiguous_flight_flag"] = df_bookings["flight_id"].isin(ambiguous_flight_ids)
df_bookings["ambiguous_passenger_flag"] = df_bookings["passenger_id"].isin(
    ambiguous_passenger_ids
)

print()
print("Bookings referencing a flight_id shared by 2+ distinct flight events:",
      df_bookings["ambiguous_flight_flag"].sum())
print("Bookings referencing a passenger_id shared by 2+ distinct people:",
      df_bookings["ambiguous_passenger_flag"].sum())

# bookings.flight_id -> flights and bookings.passenger_id -> passengers
df_bookings["referential_integrity_flag"] = (
    orphan_flight | orphan_passenger |
    df_bookings["ambiguous_flight_flag"] | df_bookings["ambiguous_passenger_flag"]
)

# payments.booking_id -> bookings
df_payments["referential_integrity_flag"] = orphan_booking

print()
print("bookings.referential_integrity_flag:", int(df_bookings["referential_integrity_flag"].sum()))
print("payments.referential_integrity_flag:", int(df_payments["referential_integrity_flag"].sum()))

logger.info(
    f"Referential integrity: 0 orphans across all 3 relationships; "
    f"{df_bookings['referential_integrity_flag'].sum()} bookings flagged as "
    f"ambiguous (non-unique parent key); "
    f"{df_payments['referential_integrity_flag'].sum()} payments flagged as orphaned"
)


bookings.flight_id orphans: 0
bookings.passenger_id orphans: 0
payments.booking_id orphans: 0
-> No orphans on any of the three relationships; nothing to quarantine for missing parents.

Bookings referencing a flight_id shared by 2+ distinct flight events: 2
Bookings referencing a passenger_id shared by 2+ distinct people: 34

bookings.referential_integrity_flag: 36
payments.referential_integrity_flag: 0


## PII Protection

None of the assignment's required KPIs (duration, route traffic,
anomalies, airline distribution) need passenger identity fields.

In [88]:
pii_protection_plan = pd.DataFrame([
    ["first_name", "Removed", "Not required by any assignment KPI", "No"],
    ["last_name", "Removed", "Not required by any assignment KPI", "No"],
    ["email", "Removed", "Not required by any assignment KPI", "No"],
    ["phone", "Removed", "Not required by any assignment KPI", "No"],
    ["aadhaar_id", "Removed (not hashed)",
     "passenger_id already provides a safe join key; hashing Aadhaar "
     "would add sensitive-data handling for no analytical benefit", "No"],
    ["date_of_birth", "Replaced with age_band",
     "Exact DOB is directly identifying; age_band supports demographic "
     "KPIs without revealing birth date", "No (age_band retained instead)"],
    ["passport_number", "Removed",
     "Explicitly must never be exposed in Power BI; no analytical use", "No"],
    ["emergency_contact_name", "Removed", "Pure PII, no analytical value", "No"],
    ["emergency_contact_phone", "Removed", "Pure PII, no analytical value", "No"],
], columns=["field", "protection_technique", "reason", "retained_in_analytical_layer"])

pii_protection_plan

,field,protection_technique,reason,retained_in_analytical_layer
0,first_name,Removed,Not required by any assignment KPI,No
1,last_name,Removed,Not required by any assignment KPI,No
2,email,Removed,Not required by any assignment KPI,No
3,phone,Removed,Not required by any assignment KPI,No
4,aadhaar_id,Removed (not hashed),passenger_id already provides a safe join key;...,No
5,date_of_birth,Replaced with age_band,Exact DOB is directly identifying; age_band su...,No (age_band retained instead)
6,passport_number,Removed,Explicitly must never be exposed in Power BI; ...,No
7,emergency_contact_name,Removed,"Pure PII, no analytical value",No
8,emergency_contact_phone,Removed,"Pure PII, no analytical value",No


In [89]:
age_bins = [0, 18, 25, 35, 45, 60, 130]
age_labels = ["<18", "18-25", "26-35", "36-45", "46-60", "60+"]

passengers_analytical = clean_passengers[
    ["passenger_id", "gender", "age", "missing_last_name_flag"]
].copy()

passengers_analytical["age_band"] = pd.cut(
    passengers_analytical["age"], bins=age_bins, labels=age_labels, right=True
)
passengers_analytical = passengers_analytical.drop(columns=["age"])

bookings_analytical = df_bookings.drop(
    columns=["passport_number", "emergency_contact_name", "emergency_contact_phone"]
).copy()

print("passengers_analytical columns:", passengers_analytical.columns.tolist())
print("bookings_analytical columns:", bookings_analytical.columns.tolist())
print()
print(passengers_analytical["age_band"].value_counts(dropna=False).sort_index())

passengers_analytical columns: ['passenger_id', 'gender', 'missing_last_name_flag', 'age_band']
bookings_analytical columns: ['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'seat_number', 'missing_status_flag', 'invalid_status_flag', 'ambiguous_flight_flag', 'ambiguous_passenger_flag', 'referential_integrity_flag']

age_band
<18      215
18-25     80
26-35    111
36-45     95
46-60    168
60+      295
Name: count, dtype: int64


### Save Task 2 outputs

In [90]:
bookings_analytical.to_csv(PROCESSED_DIR / "clean_bookings.csv", index=False)
df_payments.to_csv(PROCESSED_DIR / "clean_payments.csv", index=False)
passengers_analytical.to_csv(PROCESSED_DIR / "clean_passengers.csv", index=False)
passenger_anomalies.drop(columns=["aadhaar_id", "aadhaar_id_padded"]).to_csv(
    PROCESSED_DIR / "passenger_anomalies.csv", index=False
)

print("Task 2 summary")
print("=" * 40)
print("Bookings   :", len(bookings_analytical),
      "| flagged:", int(bookings_analytical["referential_integrity_flag"].sum()))
print("Payments   :", len(df_payments),
      "| missing/invalid amount:",
      int(df_payments["missing_amount_flag"].sum() + df_payments["invalid_amount_flag"].sum()))
print("Passengers :", len(passengers_analytical), "clean +",
      len(passenger_anomalies), "quarantined")

logger.info("Task 2 completed for bookings, payments, passengers")

Task 2 summary
Bookings   : 1000 | flagged: 36
Payments   : 1000 | missing/invalid amount: 78
Passengers : 964 clean + 75 quarantined


# Task 3: Data Modelling, Storage, and Business KPIs

A KPI(Key Performance Indicator) is a measurable value that shows how well a company, team, or person reaches important goals.

In [91]:
clean_flights.head()

,flight_event_key,flight_id,airline,source,destination,route,departure_time,arrival_time,departure_date,flight_duration_minutes,...,missing_departure_time_flag,missing_arrival_time_flag,timestamp_order_flag,sj192_anomaly_flag,duration_anomaly_flag,duration_discrepancy_flag,referential_integrity_flag,duration_kpi_eligible_flag,anomaly_flag,anomaly_reason
0,SpiceJet|SJ010|CCU|MAA|2026-04-20 23:38:41.701,SJ010,SpiceJet,CCU,MAA,CCU → MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,2026-04-20,174.0,...,False,False,False,False,False,False,False,True,False,Valid
1,Air India|AI155|BOM|CCU|2026-04-20 23:35:41.703,AI155,Air India,BOM,CCU,BOM → CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,2026-04-20,108.0,...,False,False,False,False,False,False,False,True,False,Valid
2,Vistara|UK094|BOM|CCU|2026-04-20 23:26:41.702,UK094,Vistara,BOM,CCU,BOM → CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,2026-04-20,105.0,...,False,False,False,False,False,False,False,True,False,Valid
3,Air India|AI245|BOM|CCU|2026-04-20 23:07:41.704,AI245,Air India,BOM,CCU,BOM → CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,2026-04-20,156.0,...,False,False,False,False,False,False,False,True,False,Valid
4,Air India|AI192|MAA|BOM|2026-04-20 23:05:41.703,AI192,Air India,MAA,BOM,MAA → BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,2026-04-20,299.0,...,False,False,False,False,False,False,False,True,False,Valid


In [92]:
print("Clean records:", len(clean_flights))
print("Columns:", clean_flights.columns.tolist())

Clean records: 1004
Columns: ['flight_event_key', 'flight_id', 'airline', 'source', 'destination', 'route', 'departure_time', 'arrival_time', 'departure_date', 'flight_duration_minutes', 'calculated_duration_minutes', 'source_duration_minutes', 'duration_difference', 'overnight_flag', 'duplicate_flight_id_flag', 'invalid_flight_id_flag', 'missing_airline_flag', 'missing_source_flag', 'missing_destination_flag', 'missing_departure_time_flag', 'missing_arrival_time_flag', 'timestamp_order_flag', 'sj192_anomaly_flag', 'duration_anomaly_flag', 'duration_discrepancy_flag', 'referential_integrity_flag', 'duration_kpi_eligible_flag', 'anomaly_flag', 'anomaly_reason']


## Data Model



In [93]:
fact_flights = clean_flights.copy()

dim_airline = pd.DataFrame({"airline": sorted(fact_flights["airline"].unique())})

dim_route = fact_flights[["route", "source", "destination"]].drop_duplicates().reset_index(drop=True)

dim_date = pd.DataFrame({"departure_date": sorted(fact_flights["departure_date"].unique())})
dim_date["departure_date"] = pd.to_datetime(dim_date["departure_date"])
dim_date["year"] = dim_date["departure_date"].dt.year
dim_date["month"] = dim_date["departure_date"].dt.month
dim_date["day"] = dim_date["departure_date"].dt.day
dim_date["day_name"] = dim_date["departure_date"].dt.day_name()
dim_date["is_weekend"] = dim_date["day_name"].isin(["Saturday", "Sunday"])

dim_passenger = passengers_analytical.copy()

fact_bookings = bookings_analytical.copy()
fact_payments = df_payments.copy()

print("dim_airline    :", dim_airline.shape)
print("dim_route      :", dim_route.shape)
print("dim_date       :", dim_date.shape)
print("dim_passenger  :", dim_passenger.shape)
print("fact_flights   :", fact_flights.shape)
print("fact_bookings  :", fact_bookings.shape)
print("fact_payments  :", fact_payments.shape)

dim_airline    : (4, 1)
dim_route      : (30, 3)
dim_date       : (4, 6)
dim_passenger  : (964, 4)
fact_flights   : (1004, 29)
fact_bookings  : (1000, 11)
fact_payments  : (1000, 7)


## Business KPIs

The four KPIs the assignment explicitly requires, plus a few additional ones
justified by what the model actually supports.

KPI 1: Total Flights

In [94]:
total_flights = len(clean_flights)

print("Total Flights:", total_flights)

Total Flights: 1004


KPI 2: Total Airlines

In [95]:
total_airlines = clean_flights["airline"].nunique()

print("Total Airlines:", total_airlines)

Total Airlines: 4


In [96]:
print("Total airlines:", clean_flights["airline"].nunique())
print("Null airlines:", clean_flights["airline"].isna().sum())

Total airlines: 4
Null airlines: 0


In [97]:
clean_flights["airline"].unique()

<ArrowStringArray>
['SpiceJet', 'Air India', 'Vistara', 'IndiGo']
Length: 4, dtype: str

### KPI: Average Flight Duration

In [98]:
# SJ192 (and any other record failing duration_kpi_eligible_flag) is
# excluded here so the impossible-timestamp anomaly cannot distort the KPI.
duration_kpi_data = fact_flights[fact_flights["duration_kpi_eligible_flag"]]

avg_duration_overall = duration_kpi_data["calculated_duration_minutes"].mean()

avg_duration_by_airline = (
    duration_kpi_data.groupby("airline")["calculated_duration_minutes"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
)

avg_duration_by_route = (
    duration_kpi_data.groupby("route")["calculated_duration_minutes"]
    .agg(avg_duration_minutes="mean", flights="count")
    .round(1)
    .sort_values("avg_duration_minutes", ascending=False)
)

print(
    "Duration KPI eligible records:", len(duration_kpi_data),
    "of", len(fact_flights)
)
print("Average flight duration (overall):", round(avg_duration_overall, 1), "minutes")
print()
print("By airline:")
print(avg_duration_by_airline)
print()
print("By route (top 10 longest average duration):")
print(avg_duration_by_route.head(10))


Duration KPI eligible records: 1004 of 1004
Average flight duration (overall): 164.5 minutes

By airline:
airline
Air India    165.4
IndiGo       164.9
Vistara      164.3
SpiceJet     163.2
Name: calculated_duration_minutes, dtype: float64

By route (top 10 longest average duration):
           avg_duration_minutes  flights
route                                   
BLR → MAA                 187.3       16
HYD → DEL                 185.4       42
BOM → MAA                 183.4       27
BOM → HYD                 180.8       26
DEL → MAA                 178.8       23
DEL → BOM                 178.1       28
MAA → BOM                 176.8       21
BOM → BLR                 176.5       23
DEL → HYD                 174.8       54
MAA → BLR                 172.8       65


### KPI: Route-wise Traffic

In [99]:
route_traffic = (
    fact_flights.groupby("route")
    .size()
    .rename("flight_count")
    .sort_values(ascending=False)
)

print("Total routes:", route_traffic.shape[0])
print()
print(route_traffic)

Total routes: 30

route
BOM → CCU    90
CCU → DEL    72
MAA → BLR    65
BLR → BOM    60
HYD → MAA    57
DEL → HYD    54
HYD → DEL    42
BOM → DEL    39
CCU → BOM    33
DEL → BLR    29
DEL → BOM    28
BOM → MAA    27
BOM → HYD    26
HYD → BOM    26
MAA → DEL    26
HYD → CCU    26
HYD → BLR    26
DEL → CCU    26
MAA → CCU    24
CCU → MAA    24
BOM → BLR    23
DEL → MAA    23
BLR → CCU    21
CCU → HYD    21
MAA → BOM    21
MAA → HYD    21
CCU → BLR    20
BLR → DEL    19
BLR → HYD    19
BLR → MAA    16
Name: flight_count, dtype: int64


### KPI: Delays / Anomalies

In [100]:
print("Flight-level anomalies:", len(anomaly_flights), "of", len(df))
print(anomaly_flights["anomaly_reason"].value_counts())
print()

print("Flight-level anomaly breakdown (by flag):")
print("  sj192_anomaly_flag       :", int(df["sj192_anomaly_flag"].sum()))
print("  timestamp_order_flag     :", int(df["timestamp_order_flag"].sum()))
print("  duration_anomaly_flag    :", int(df["duration_anomaly_flag"].sum()))
print("  duration_discrepancy_flag:", int(df["duration_discrepancy_flag"].sum()))
print("  referential_integrity_flag (flights):", int(df["referential_integrity_flag"].sum()))
print()

print("Booking-level referential anomalies (ambiguous flight/passenger reference):")
print("  ambiguous_flight_flag   :", int(fact_bookings["ambiguous_flight_flag"].sum()))
print("  ambiguous_passenger_flag:", int(fact_bookings["ambiguous_passenger_flag"].sum()))
print("  missing/invalid status  :",
      int(fact_bookings["missing_status_flag"].sum() + fact_bookings["invalid_status_flag"].sum()))
print("  referential_integrity_flag (bookings):", int(fact_bookings["referential_integrity_flag"].sum()))
print()

print("Payment-level anomalies:")
print("  missing_amount_flag :", int(fact_payments["missing_amount_flag"].sum()))
print("  invalid_amount_flag :", int(fact_payments["invalid_amount_flag"].sum()))
print("  referential_integrity_flag (payments):", int(fact_payments["referential_integrity_flag"].sum()))
print()

print("Passenger-level anomalies:")
print("  ambiguous passenger_id records quarantined:", len(passenger_anomalies))
print("  missing_last_name_flag                     :",
      int(dim_passenger["missing_last_name_flag"].sum()))


Flight-level anomalies: 3 of 1005
anomaly_reason
Duplicate Flight ID, Referential Integrity Failure      2
SJ192 Impossible Timestamp Anomaly, Invalid Duration    1
Name: count, dtype: int64

Flight-level anomaly breakdown (by flag):
  sj192_anomaly_flag       : 1
  timestamp_order_flag     : 1
  duration_anomaly_flag    : 1
  duration_discrepancy_flag: 0
  referential_integrity_flag (flights): 2

Booking-level referential anomalies (ambiguous flight/passenger reference):
  ambiguous_flight_flag   : 2
  ambiguous_passenger_flag: 34
  missing/invalid status  : 75
  referential_integrity_flag (bookings): 36

Payment-level anomalies:
  missing_amount_flag : 48
  invalid_amount_flag : 30
  referential_integrity_flag (payments): 0

Passenger-level anomalies:
  ambiguous passenger_id records quarantined: 75
  missing_last_name_flag                     : 0


### KPI: Distribution of Flights by Airline

In [101]:
airline_distribution = fact_flights["airline"].value_counts()
airline_distribution_pct = (
    airline_distribution / airline_distribution.sum() * 100
).round(1)

airline_distribution_table = pd.DataFrame({
    "flight_count": airline_distribution,
    "percentage": airline_distribution_pct
})

airline_distribution_table

,flight_count,percentage
airline,,
IndiGo,273,27.2
Air India,255,25.4
SpiceJet,246,24.5
Vistara,230,22.9


### Additional KPIs (justified by the model)

- **Overnight share** — how much of the schedule crosses midnight (useful for
  crew/ops planning context).
- **Booking status mix** — CONFIRMED vs CANCELLED vs PENDING vs Unknown.
- **Revenue KPIs** — total and average payment amount by method, computed
  only from valid numeric amounts (missing/invalid are excluded, not zeroed).
- **Passenger demographics** — gender and age-band mix, from the PII-safe
  passenger dimension.

In [102]:
overnight_share = (fact_flights["overnight_flag"].mean() * 100).round(1)
print("Overnight flight share:", overnight_share, "%")
print()

booking_status_mix = fact_bookings["status"].value_counts()
print("Booking status mix:")
print(booking_status_mix)
print()

valid_payments = fact_payments[fact_payments["amount"].notna()]
revenue_by_method = valid_payments.groupby("payment_method")["amount"].agg(
    total_amount="sum", avg_amount="mean", payment_count="count"
).round(2)
print("Revenue by payment method (valid payments only):")
print(revenue_by_method)
print()
print("Total recorded revenue (valid payments only):", round(valid_payments["amount"].sum(), 2))
print()

print("Passenger gender mix:")
print(dim_passenger["gender"].value_counts(dropna=False))
print()
print("Passenger age-band mix:")
print(dim_passenger["age_band"].value_counts(dropna=False).sort_index())

Overnight flight share: 12.2 %

Booking status mix:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
Unknown       75
Name: count, dtype: int64[pyarrow]

Revenue by payment method (valid payments only):
                total_amount  avg_amount  payment_count
payment_method                                         
CARD              2400812.04     8002.71            300
NETBANKING        2365644.17     8129.36            291
UPI               2618686.77     7911.44            331

Total recorded revenue (valid payments only): 7385142.98

Passenger gender mix:
gender
M    509
F    455
Name: count, dtype: int64[pyarrow]

Passenger age-band mix:
age_band
<18      215
18-25     80
26-35    111
36-45     95
46-60    168
60+      295
Name: count, dtype: int64


# Task 4: Power BI Preparation

Power BI does the KPI calculation itself (measures/visuals) — this notebook's
job is to hand it a clean, joinable star schema, not pre-baked numbers. All
tables below are exported as-is (no KPI values hardcoded into the export);
the KPI cells above exist to validate the model, not to freeze results into
the files.

Export folder: `data/processed/powerbi/`

| File | Role | Power BI relationship |
|---|---|---|
| `fact_flights.csv` | Fact (flight events) | → `dim_airline`, `dim_route`, `dim_date` |
| `fact_bookings.csv` | Fact (bookings) | → `fact_flights.flight_id`, `dim_passenger.passenger_id` |
| `fact_payments.csv` | Fact (payments) | → `fact_bookings.booking_id` |
| `dim_airline.csv` | Dimension | one row per airline |
| `dim_route.csv` | Dimension | one row per source→destination pair |
| `dim_date.csv` | Dimension | one row per departure date, mark as Date Table |
| `dim_passenger.csv` | Dimension | PII-safe passenger attributes only |
| `flight_anomalies.csv` | Anomaly table | for an Anomaly Insights page, not joined into the star |
| `passenger_anomalies.csv` | Anomaly table | for review only, not joined into the star |



In [103]:
POWERBI_DIR = PROCESSED_DIR / "powerbi"
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    "fact_flights.csv": fact_flights,
    "fact_bookings.csv": fact_bookings,
    "fact_payments.csv": fact_payments,
    "dim_airline.csv": dim_airline,
    "dim_route.csv": dim_route,
    "dim_date.csv": dim_date,
    "dim_passenger.csv": dim_passenger,
    "flight_anomalies.csv": anomaly_flights,
    "passenger_anomalies.csv": passenger_anomalies.drop(columns=["aadhaar_id", "aadhaar_id_padded"]),
}

for filename, table in exports.items():
    table.to_csv(POWERBI_DIR / filename, index=False)

logger.info(f"Power BI export written to {POWERBI_DIR}: {list(exports.keys())}")

print("Exported to:", POWERBI_DIR)
for filename, table in exports.items():
    print(f"  {filename:<28} {table.shape}")

Exported to: C:\Users\krkun\OneDrive\Desktop\Neo Stats\data\processed\powerbi
  fact_flights.csv             (1004, 29)
  fact_bookings.csv            (1000, 11)
  fact_payments.csv            (1000, 7)
  dim_airline.csv              (4, 1)
  dim_route.csv                (30, 3)
  dim_date.csv                 (4, 6)
  dim_passenger.csv            (964, 4)
  flight_anomalies.csv         (3, 39)
  passenger_anomalies.csv      (75, 10)


In [104]:
logger.info("Pipeline completed successfully")

print("=" * 50)
print("PIPELINE COMPLETE")
print("=" * 50)
print(f"Flights   : {len(fact_flights)} clean, {len(anomaly_flights)} anomalies")
print(f"Bookings  : {len(fact_bookings)}, {int(fact_bookings['referential_integrity_flag'].sum())} flagged")
print(f"Payments  : {len(fact_payments)}, {int(fact_payments['missing_amount_flag'].sum() + fact_payments['invalid_amount_flag'].sum())} missing/invalid amount")
print(f"Passengers: {len(dim_passenger)} clean, {len(passenger_anomalies)} quarantined")
print(f"Power BI export: {POWERBI_DIR}")

PIPELINE COMPLETE
Flights   : 1004 clean, 3 anomalies
Bookings  : 1000, 36 flagged
Payments  : 1000, 78 missing/invalid amount
Passengers: 964 clean, 75 quarantined
Power BI export: C:\Users\krkun\OneDrive\Desktop\Neo Stats\data\processed\powerbi


## Final Validation

A concise set of checks on the flight-level pipeline: record counts through
deduplication, each required anomaly flag, KPI eligibility, and an explicit
check that the two distinct `6F250` flight events and the untouched `SJ192`
timestamps both survived the pipeline.


In [105]:
print("=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

total_after_dedup = len(df)
duplicate_flight_id_count = int(df["duplicate_flight_id_flag"].sum())
overnight_count = int(df["overnight_flag"].sum())
sj192_count = int(df["sj192_anomaly_flag"].sum())
duration_anomaly_count = int(df["duration_anomaly_flag"].sum())
duration_discrepancy_count = int(df["duration_discrepancy_flag"].sum())
referential_integrity_failures = (
    int(df["referential_integrity_flag"].sum())
    + int(fact_bookings["referential_integrity_flag"].sum())
    + int(fact_payments["referential_integrity_flag"].sum())
)
kpi_eligible_count = int(df["duration_kpi_eligible_flag"].sum())
overall_anomaly_count = int(df["anomaly_flag"].sum())

print("Total flight records after exact duplicate removal:", total_after_dedup)
print("Duplicate flight ID records (flag=True)            :", duplicate_flight_id_count)
print("Overnight flights                                   :", overnight_count)
print("SJ192 anomalies                                     :", sj192_count)
print("Duration anomalies                                  :", duration_anomaly_count)
print("Duration discrepancies                              :", duration_discrepancy_count)
print("Referential integrity failures (flights+bookings+payments):",
      referential_integrity_failures)
print("Duration KPI-eligible records                       :",
      kpi_eligible_count, "of", total_after_dedup)
print("Overall anomalies (anomaly_flag)                    :", overall_anomaly_count)
print()

# --- 6F250: both distinct flight events must be retained and disambiguated ---
sixf250 = df[df["flight_id"] == "6F250"]
print("6F250 records retained:", len(sixf250))
print(sixf250[["flight_event_key", "flight_id", "source", "destination", "departure_time"]])

assert len(sixf250) == 2, "Expected both 6F250 flight events to be retained"
assert sixf250["source"].nunique() == 2, \
    "Expected the two 6F250 events to have different sources"
assert sixf250["flight_event_key"].is_unique, \
    "flight_event_key must disambiguate the two 6F250 events"
assert sixf250["flight_id"].isin(clean_flights["flight_id"]).all(), \
    "Both 6F250 events must remain in the flight fact table (clean_flights)"

# --- SJ192: timestamps must be untouched, flagged, and excluded from the KPI ---
sj192_row = df[df["flight_id"] == "SJ192"]
assert len(sj192_row) == 1, "Expected exactly one SJ192 record"
assert bool(sj192_row["sj192_anomaly_flag"].iloc[0]), "SJ192 must be flagged"
assert bool(sj192_row["anomaly_flag"].iloc[0]), "SJ192 must count as an overall anomaly"
assert not bool(sj192_row["duration_kpi_eligible_flag"].iloc[0]), \
    "SJ192 must be excluded from duration KPI eligibility"
assert sj192_row["flight_id"].iloc[0] not in clean_flights["flight_id"].values, \
    "SJ192 should not be treated as a usable flight fact"
assert (sj192_row["arrival_time"].iloc[0] < sj192_row["departure_time"].iloc[0]), \
    "SJ192 timestamps must remain in their original (impossible) order -- do not correct them"
assert not bool(sj192_row["overnight_flag"].iloc[0]), \
    "SJ192 must not be treated as a normal overnight flight"

print()
print("All validation checks passed.")


FINAL VALIDATION
Total flight records after exact duplicate removal: 1005
Duplicate flight ID records (flag=True)            : 2
Overnight flights                                   : 122
SJ192 anomalies                                     : 1
Duration anomalies                                  : 1
Duration discrepancies                              : 0
Referential integrity failures (flights+bookings+payments): 38
Duration KPI-eligible records                       : 1004 of 1005
Overall anomalies (anomaly_flag)                    : 3

6F250 records retained: 2
                                 flight_event_key flight_id source  \
253  IndiGo|6F250|DEL|BLR|2026-04-20 03:26:41.701     6F250    DEL   
270  IndiGo|6F250|CCU|BLR|2026-04-20 02:23:41.702     6F250    CCU   

    destination          departure_time  
253         BLR 2026-04-20 03:26:41.701  
270         BLR 2026-04-20 02:23:41.702  

All validation checks passed.
